# Review triage for a phone-accessories retailer

**BC3415 Individual Assignment, Option A.** A retailer receives thousands of reviews a day, as text and as customer photos. This notebook:
1. classifies each review as **positive / neutral / negative**,
2. detects **what went wrong** (defect, counterfeit, wrong item, shipping damage, performance), and
3. combines the text, metadata and photo signals into a **ranked queue of reviews that point to problematic orders**.

Sections: §1 Data audit · §2 Preprocessing · §3 Models · §4 Explainability · §5 Combining signals · §6 Live demo · §7 Limitations.
Reusable logic (cleaning, issue tagging, flag score) lives in `triage.py`. On the first run, two pretrained models (~1.3 GB) are downloaded.

In [ ]:
import io
import json
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, f1_score,
                             precision_recall_fscore_support)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          CLIPModel, CLIPProcessor)
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import triage

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

LABELS = ["negative", "neutral", "positive"]
COLORS = {"negative": "#eb6834", "neutral": "#8a8984", "positive": "#2a78d6"}
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.axisbelow": True,
    "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "text.color": "#0b0b0b",
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e", "font.size": 10,
})
pd.set_option("display.max_colwidth", 160)
START = time.time()

## 1. Data audit

**What:** 9,000 reviews from the *Cell Phones & Accessories* category of Amazon Reviews 2023 (McAuley Lab), with 3,000 per class. They were drawn from a random pool of ~231k reviews by `scripts/build_sample.py`.
**Why:** the labels come from **star ratings, not human annotation**, so we test how trustworthy they are before trusting any metric.
**Look at:** the class imbalance, titles that leak the label, duplicates with conflicting labels, and reviews whose text contradicts their stars.

In [ ]:
df = pd.read_csv("data/sample.csv")
df["review_id"] = df.index
with open("data/pool_stats.json") as f:
    pool = json.load(f)

print(f"Pool: {pool['pool_reviews']:,} reviews read from {pool['chunks_read']} random slices "
      f"of the {pool['file_bytes'] / 1e9:.1f} GB {pool['category']} file")
print("Balanced sample:", df.shape)
df[["rating", "label", "title", "text", "n_images", "helpful_vote", "verified_purchase"]].head()

In [ ]:
natural = pd.Series(pool["pool_label_counts"]).reindex(LABELS)
balanced = df["label"].value_counts().reindex(LABELS)
shares = pd.DataFrame({
    "natural pool (%)": (100 * natural / natural.sum()).round(1),
    "balanced sample (%)": (100 * balanced / balanced.sum()).round(1),
})
print(shares)

fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, col in zip(axes, shares.columns):
    bars = ax.bar(LABELS, shares[col], color=[COLORS[l] for l in LABELS], width=0.6)
    ax.bar_label(bars, fmt="%.1f%%", padding=2)
    ax.set_title(col.replace(" (%)", ""))
axes[0].set_ylabel("% of reviews")
plt.tight_layout()
plt.show()

In the wild, **neutral (3★) reviews are only 7.3%** of this category and positives are 73.5%. A model trained on that mix could score about 74% accuracy by always predicting "positive" while never finding a neutral review. We therefore **balance to 3,000 per class** and use **macro-F1** as the headline metric, which weights every class equally.
Trade-off: on a balanced test set, precision for the rare neutral class looks better than it would on live traffic. Recall does not depend on class mix, so it carries over.

In [ ]:
title = df["title"].fillna("")
text = df["text"].fillna("")
auto_title = title.str.strip().str.match(triage.AUTO_TITLE_PATTERN, case=False)
n_words = (title + " " + text).str.split().str.len()

audit = pd.Series({
    "missing or empty text": (text.str.strip() == "").sum(),
    "missing title": df["title"].isna().sum(),
    "text containing HTML <br /> tags": text.str.contains("<br", regex=False).sum(),
    "auto-generated 'N Stars' title": auto_title.sum(),
    "fewer than 5 words (title + text)": (n_words < 5).sum(),
    "same title+text appears more than once": (title + " " + text).duplicated(keep=False).sum(),
}, name="rows")
audit.to_frame()

In [ ]:
# Auto titles vs the actual star rating
pd.crosstab(title[auto_title].str.strip().str.title(), df.loc[auto_title, "rating"])

**Leakage found.** When a customer leaves the title blank, Amazon fills it with "Five Stars", "One Star" and so on. These titles **spell out the label**: 896 rows (~10%) have one, and the table shows they agree with the rating almost perfectly. If we left them in, the model would score well by reading the answer. §2 removes them from the model input. The few mismatches (e.g. "Five Stars" on a 1★ review) show the star field itself is not always reliable.

In [ ]:
# Candidate label noise: short reviews where VADER sentiment strongly disagrees with the stars.
sia = SentimentIntensityAnalyzer()
clean = text.apply(triage.clean_text)
compound = clean.apply(lambda t: sia.polarity_scores(t)["compound"])
words = clean.str.split().str.len()
low_star_happy = (df["rating"] <= 2) & (compound > 0.7) & words.between(6, 45)
high_star_angry = (df["rating"] >= 4) & (compound < -0.5) & words.between(4, 45)
print("candidates: 1-2 stars but positive text:", low_star_happy.sum(),
      "| 4-5 stars but negative text:", high_star_angry.sum())

# We read the candidates by hand. Many are VADER mistakes (e.g. "my battery was dead by 10am...
# this fixed it"), but these five are real contradictions between stars and text:
NOISE_IDS = [2711, 3112, 5924, 3350, 8147]
df.loc[NOISE_IDS, ["rating", "label", "title", "text"]]

**The labels are noisy.** #2711 gives 1★ to a review that says "Perfect… Recommend". #5924 ("Stopped working… would not buy again") and #3350 ("Product Failure… Very disappointed") are 5★. #8147 is 5★ because the seller replaced a cable, yet the text reports a defect, which is still worth flagging for triage. Some errors in §3 will therefore be *the label's fault, not the model's*, and no model can reach 100%. We also found reviews filed under the wrong product (#7317 reviews a Star Wars toy inside Cell Phones), a known side effect of Amazon merging product listings.

## 2. Preprocessing

**What:** build one model input per review, **title + text**, with HTML tags and entities removed and the leaky auto-titles dropped. Then remove empty and duplicate inputs and make a stratified 80/20 train/test split.
**Why:** duplicate texts landing in both train and test would let a model "remember" answers, and identical texts with different stars are pure label noise.
**Look at:** how many rows each step removes, and the duplicate groups with conflicting labels.

In [ ]:
df["input"] = [triage.make_input(t, x) for t, x in zip(df["title"], df["text"])]
df["n_words"] = df["input"].str.split().str.len()
rows = {"start": len(df)}

df = df[df["input"] != ""]
rows["after dropping empty inputs"] = len(df)

key = df["input"].str.lower()
dups = df[key.duplicated(keep=False)]
groups = dups.groupby(key[dups.index])["label"].agg(copies="size", labels=lambda s: ", ".join(sorted(set(s))))
print(f"{len(dups)} rows share their input with another row ({len(groups)} groups); "
      f"{groups['labels'].str.contains(',').sum()} groups carry conflicting labels")
display(groups.sort_values("copies", ascending=False).head(8))

df = df[~key.duplicated(keep="first")].copy()
rows["after dropping duplicate inputs"] = len(df)
print(pd.Series(rows, name="rows"))
print(df["label"].value_counts())

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=SEED)
train_df, test_df = train_df.copy(), test_df.copy()
print("train:", train_df["label"].value_counts().to_dict())
print("test: ", test_df["label"].value_counts().to_dict())
print(f"short reviews (<5 words) kept: {(df['n_words'] < 5).sum()} - flagged, not removed, because real traffic has them")

## 3. Models

**What:** three approaches of increasing cost. (1) **VADER**, a rule-based lexicon with no training. (2) **TF-IDF (1–2-grams) + Logistic Regression**, trained on our 80%. (3) **Pretrained BERT** (`nlptown/bert-base-multilingual-uncased-sentiment`), used for inference only. It predicts 1–5★, and we sum its probabilities into our 3 classes. We also try (4) **averaging the LR and BERT probabilities**.
**Why:** to see whether training on in-domain data or a large pretrained model is worth its cost.
**Look at:** macro-F1 (headline) and neutral recall, the hardest class.

In [ ]:
def vader_label(text):
    compound = sia.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"

test_df["pred_vader"] = test_df["input"].apply(vader_label)

In [ ]:
lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
    ("lr", LogisticRegression(max_iter=2000)),
])
search = GridSearchCV(lr_pipe, {"lr__C": [0.5, 1, 2, 4, 8]}, scoring="f1_macro", cv=5, n_jobs=2)  # 2 workers keeps RAM low
search.fit(train_df["input"], train_df["label"])
lr_model = search.best_estimator_
print("best C:", search.best_params_["lr__C"], "| 5-fold CV macro-F1:", round(search.best_score_, 3))
print("vocabulary size:", len(lr_model.named_steps["tfidf"].vocabulary_))

assert list(lr_model.classes_) == LABELS
lr_test_probs = lr_model.predict_proba(test_df["input"])
test_df["pred_lr"] = np.array(LABELS)[lr_test_probs.argmax(axis=1)]

In [ ]:
BERT_NAME = "nlptown/bert-base-multilingual-uncased-sentiment"
bert_tok = AutoTokenizer.from_pretrained(BERT_NAME)
bert = AutoModelForSequenceClassification.from_pretrained(BERT_NAME).eval()

def bert_probs(texts, batch_size=32):
    # Returns [P(negative), P(neutral), P(positive)] per text: 1+2 stars, 3 stars, 4+5 stars.
    texts = list(texts)
    order = np.argsort([len(t) for t in texts])  # batch similar lengths together: less padding, ~3x faster on CPU
    probs = np.zeros((len(texts), 3))
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        batch = bert_tok([texts[j] for j in idx], padding=True, truncation=True,
                         max_length=256, return_tensors="pt")
        with torch.no_grad():
            stars = torch.softmax(bert(**batch).logits, dim=1).numpy()
        probs[idx] = np.column_stack([stars[:, 0] + stars[:, 1], stars[:, 2], stars[:, 3] + stars[:, 4]])
    return probs

t0 = time.time()
bert_test_probs = bert_probs(test_df["input"].tolist())
test_df["pred_bert"] = np.array(LABELS)[bert_test_probs.argmax(axis=1)]
print(f"BERT inference on {len(test_df)} test reviews: {time.time() - t0:.0f}s")

ens_test_probs = (lr_test_probs + bert_test_probs) / 2
test_df["pred_ensemble"] = np.array(LABELS)[ens_test_probs.argmax(axis=1)]

In [ ]:
def score_row(name, y_true, y_pred):
    p, r, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    row = {"model": name,
           "macro_f1": f1_score(y_true, y_pred, labels=LABELS, average="macro"),
           "accuracy": accuracy_score(y_true, y_pred)}
    for i, label in enumerate(LABELS):
        row[f"{label}_precision"] = p[i]
        row[f"{label}_recall"] = r[i]
    return row

PRED_COLS = {"VADER (lexicon)": "pred_vader", "TF-IDF + LogReg": "pred_lr",
             "BERT nlptown (pretrained)": "pred_bert", "LogReg + BERT average": "pred_ensemble"}
results = pd.DataFrame([score_row(name, test_df["label"], test_df[col]) for name, col in PRED_COLS.items()])
results = results.sort_values("macro_f1", ascending=False).round(3)
os.makedirs("reports", exist_ok=True)
results.to_csv("reports/results.csv", index=False)
results

In [ ]:
best_two = results["model"].head(2).tolist()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name in zip(axes, best_two):
    ConfusionMatrixDisplay.from_predictions(test_df["label"], test_df[PRED_COLS[name]], labels=LABELS,
                                            ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
    ax.grid(False)
plt.tight_layout()
plt.show()

## 4. Explainability

**What:** (a) the n-grams that push the Logistic Regression hardest towards each class, read straight from its coefficients. (b) Six test reviews the best model got wrong, each tagged by hand as **label noise** (the stars are wrong) or **model error** (the model is wrong).
**Why:** a retailer will only act on a model whose reasons make sense, and error analysis tells us whether to improve the model or the data.
**Look at:** whether the top words are sensible product language, and how many "errors" are really bad labels.

In [ ]:
vec = lr_model.named_steps["tfidf"]
clf = lr_model.named_steps["lr"]
terms = np.array(vec.get_feature_names_out())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, i, label in zip(axes, range(3), clf.classes_):
    top = np.argsort(clf.coef_[i])[-15:]
    ax.barh(terms[top], clf.coef_[i][top], color=COLORS[label], height=0.7)
    ax.set_title(f"{label}: top 15 n-grams")
    ax.set_xlabel("LogReg coefficient")
plt.tight_layout()
plt.show()

In [ ]:
best_name = results["model"].iloc[0]
best_col = PRED_COLS[best_name]
wrong = test_df[test_df[best_col] != test_df["label"]]
print(f"{best_name}: {len(wrong)} of {len(test_df)} test reviews misclassified")
examples = wrong.groupby("label").sample(n=2, random_state=SEED)
examples[["review_id", "rating", "label", best_col, "input"]]

## 5. Combining signals: issue tags + photos → flag queue

**What:** we treat the 1,800-ish test reviews as "today's intake" and score each one on four signals. (1) The model's P(negative). (2) The worst **issue** found by a keyword tagger (`triage.ISSUE_KEYWORDS`). (3) **Helpful votes**: other shoppers agree. (4) A **CLIP** zero-shot score for whether the customer photo shows a damaged or wrong product.
**Why:** sentiment alone says *a customer is unhappy*, while the queue should say *which orders need action and why*.
**Look at:** the top-20 queue. The weights in `triage.WEIGHTS` are **business assumptions** (no ground-truth "problem order" label exists), so the queue is a ranking aid, not a measured classifier.

In [ ]:
test_df["issues"] = test_df["input"].apply(triage.tag_issues)
issue_counts = (test_df.explode("issues")
                .groupby(["issues", "label"]).size().unstack(fill_value=0).reindex(columns=LABELS))
issue_counts

In [ ]:
CLIP_NAME = "openai/clip-vit-base-patch32"
clip = CLIPModel.from_pretrained(CLIP_NAME).eval()
clip_proc = CLIPProcessor.from_pretrained(CLIP_NAME)
PROMPTS = ["a photo of a damaged, broken or wrong product", "a photo of an intact product in good condition"]

def load_image(url, cache_path=None):
    if cache_path and os.path.exists(cache_path):
        return Image.open(cache_path).convert("RGB")
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    image = Image.open(io.BytesIO(response.content)).convert("RGB")
    if cache_path:
        image.save(cache_path)
    return image

def clip_damage_prob(image):
    inputs = clip_proc(text=PROMPTS, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        probs = clip(**inputs).logits_per_image.softmax(dim=1)
    return probs[0, 0].item()   # probability of the "damaged" prompt

os.makedirs("data/images", exist_ok=True)
with_photo = test_df[test_df["n_images"] > 0].head(100)
damage, images = {}, {}
for rid, urls in zip(with_photo["review_id"], with_photo["image_urls"]):
    try:
        images[rid] = load_image(urls.split("|")[0], cache_path=f"data/images/{rid}.jpg")
        damage[rid] = clip_damage_prob(images[rid])
    except Exception as err:
        print("skipped", rid, err)
test_df["image_damage"] = test_df["review_id"].map(damage)
print(f"scored {len(damage)} customer photos")
test_df.groupby("label")["image_damage"].agg(["count", "mean"]).round(3)

In [ ]:
neg_photos = test_df[test_df["image_damage"].notna() & (test_df["label"] == "negative")]
show = neg_photos.sort_values("image_damage", ascending=False)
show = pd.concat([show.head(3), show.tail(3)])
fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))
for ax, (_, r) in zip(axes, show.iterrows()):
    ax.imshow(images[r["review_id"]])
    ax.set_title(f"#{r['review_id']}  {int(r['rating'])}★\nP(damaged)={r['image_damage']:.2f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
show[["review_id", "rating", "image_damage", "input"]]

In [ ]:
TEST_PROBS = {"TF-IDF + LogReg": lr_test_probs, "BERT nlptown (pretrained)": bert_test_probs,
              "LogReg + BERT average": ens_test_probs}
test_df["p_negative"] = TEST_PROBS[best_name][:, 0]   # P(negative) from the best model in §3
queue = triage.build_queue(test_df, top_n=20)
queue[["review_id", "rating", "flag_score", "p_negative", "issues", "helpful_vote", "image_damage", "input"]] \
    .to_csv("reports/flag_queue.csv", index=False)
print("weights (business assumptions):", triage.WEIGHTS)
print("star ratings in the top-20 queue:", queue["rating"].value_counts().sort_index().to_dict())
queue[["review_id", "rating", "flag_score", "p_negative", "issues", "helpful_vote", "image_damage", "input"]].round(3)

## 6. Live demo

**What:** a Gradio app wrapping the whole pipeline. Paste a review (and optionally a photo URL) to get sentiment probabilities, issue tags and the flag score.
**Why:** shows the pipeline is reusable on new, unseen text, not only on our test set.
**Look at:** the cell prints one worked example first, then launches a public `share=True` link.

In [ ]:
import gradio as gr

def sentiment_probs(texts):
    # Same model as the headline result in §3
    return (lr_model.predict_proba(texts) + bert_probs(texts)) / 2

def analyse(review_text, image_url="", helpful_votes=0):
    text = triage.clean_text(review_text)
    probs = sentiment_probs([text])[0]
    tags = triage.tag_issues(text)
    image_damage = None
    if image_url and image_url.strip():
        try:
            image_damage = clip_damage_prob(load_image(image_url.strip()))
        except Exception:
            image_damage = None
    score = triage.flag_score(probs[0], tags, helpful_votes, image_damage)
    photo = "no photo" if image_damage is None else f"{image_damage:.2f}"
    return dict(zip(LABELS, probs.round(3).tolist())), ", ".join(tags), round(score, 3), photo

print(analyse("Arrived with a cracked screen protector and the charger stopped working after two days. Looks fake."))

demo = gr.Interface(
    fn=analyse,
    inputs=[gr.Textbox(lines=5, label="Review text"),
            gr.Textbox(label="Customer photo URL (optional)"),
            gr.Number(value=0, label="Helpful votes")],
    outputs=[gr.Label(label="Sentiment"), gr.Textbox(label="Issue tags"),
             gr.Number(label="Flag score (0-1)"), gr.Textbox(label="P(photo shows damage)")],
    examples=[["Great case, fits perfectly and feels solid.", "", 0],
              ["It's okay. Does the job but the color is duller than the picture.", "", 0],
              ["Received the wrong model and the box was crushed. Charger is clearly a knockoff.", "", 3]],
    title="Review triage demo",
)
demo.launch(share=True)

## 7. Limitations and next steps
LIMITATIONS_PLACEHOLDER

In [ ]:
print(f"Notebook run time: {(time.time() - START) / 60:.1f} min")